In [3]:
# ============================================================
# LOAD MODULES AND MODEL OUTPUTS
# ============================================================
from Imports import *
from settings import *
from Inputs import *
from Helper import *
from PlotHelper import *
from Outputs import *

# --- Load MODFLOW 6 simulation ---
sim = flopy.mf6.MFSimulation.load(sim_ws=sim_ws, verbosity_level=0)
gwf = sim.get_model(nameModel)

# --- Model arrays from DIS package ---
top     = np.array(gwf.dis.top.array,     dtype=float)
botm    = np.array(gwf.dis.botm.array,    dtype=float)
idomain = np.array(gwf.dis.idomain.array, dtype=int)

# --- Grid properties ---
xorigin = float(gwf.modelgrid.xoffset)
yorigin = float(gwf.modelgrid.yoffset)
delr    = gwf.modelgrid.delr
delc    = gwf.modelgrid.delc
nlay, nrow, ncol = idomain.shape

# --- Head output: load all timesteps ---
hds    = bf.HeadFile(os.path.join(sim_ws, f"{nameModel}.hds"))
times  = hds.get_times()
head   = np.array([hds.get_data(totim=t) for t in times], dtype=float)
dates  = pd.date_range(start=START_DATE, periods=head.shape[0], freq="MS")
months = dates

print(f"Model loaded  : {nameModel}")
print(f"Head shape    : {head.shape}")
print(f"Date range    : {dates[0].strftime('%Y-%m')} to {dates[-1].strftime('%Y-%m')}")


NameError: name 'Path' is not defined

# This notebook will check the Simulations outputs and process the outputs

## 17) Optional: listing-file budget summary

This reads the MF6 listing file and prints the flux budget and percent discrepancy.

In [ ]:

# ---------------------------------------------------------
# MODFLOW 6 budget from listing file
# ---------------------------------------------------------
lst_path = os.path.join(sim_ws, f"{nameModel}.lst")

# MF6 uses Mf6ListBudget, not MfListBudget
mf_list = flopy.utils.Mf6ListBudget(lst_path)
df_flux, df_vol = mf_list.get_dataframes(start_datetime=START_DATE, diff=True)

print("=== FLUX BUDGET (m³/day) ===")
print(df_flux.to_string())

print("\n=== PERCENT DISCREPANCY ===")
# discrepancy column is usually named 'PERCENT_DISCREPANCY' or similar
disc_cols = [c for c in df_flux.columns if "DISCREPANCY" in c.upper() or "PERCENT" in c.upper()]
if disc_cols:
    print(df_flux[disc_cols])
else:
    print("Discrepancy column not found — printing all columns:")
    print(df_flux.columns.tolist())

## 18) Optional: example post-processing

The original notebook had many exploratory post-processing cells.  
This notebook keeps one representative post-processing figure cell as a starting point.

In [ ]:

# ============================================================
# SETTINGS
# ============================================================
save_fig = True
out_fig = r"D:\Users\abolmaal\modelling\Figs\Testing_6\avg_dtw_avg_recharge_studyperiod.png"

# if head is not already loaded
if "head" not in globals():
    head_path = os.path.join(sim_ws, f"{nameModel}.hds")
    hds = bf.HeadFile(head_path)
    times = hds.get_times()
    head = np.array([hds.get_data(totim=t) for t in times], dtype=float)
    print("Loaded head array:", head.shape)

# top from model
top = np.array(gwf.dis.top.array, dtype=float)

# ============================================================
# HELPERS
# ============================================================

# ============================================================
# 1) AVERAGE DEPTH TO WATER
# ============================================================
# =========================================================
# COMPUTE DTW — with dry cell protection
# =========================================================

dtw_all = []
for i in range(head.shape[0]):
    wt = get_water_table(head[i], idomain, huge=1e29)
    dtw = np.array(top, dtype=float) - wt

    # hard clip: anything beyond physical bounds is a dry cell artifact
    # Great Lakes Basin: no cell should have DTW > total aquifer thickness (~500 m max)
    dtw = np.where(dtw > 200, np.nan, dtw)   # treat >200 m as dry/artifact
    dtw = np.where(idomain[0] <= 0, np.nan, dtw)

    dtw_all.append(dtw)

dtw_all = np.array(dtw_all, dtype=float)

# average — nanmean ignores dry-cell NaNs
avg_dtw = np.nanmean(dtw_all, axis=0)

# mask inactive cells
avg_dtw = np.where(idomain[0] > 0, avg_dtw, np.nan)

# mask lake cells
if "lake_mask_2d" in globals():
    avg_dtw = np.where(lake_mask_2d, np.nan, avg_dtw)

# ---- final stats ----
finite = avg_dtw[np.isfinite(avg_dtw)]
print("Average DTW after cleaning:")
print(f"  min  : {np.nanmin(finite):.2f} m")
print(f"  p5   : {np.percentile(finite, 5):.2f} m")
print(f"  p50  : {np.percentile(finite, 50):.2f} m")
print(f"  p95  : {np.percentile(finite, 95):.2f} m")
print(f"  max  : {np.nanmax(finite):.2f} m")
print(f"  negative cells : {int(np.sum(finite < 0))}")
print(f"  cells > 50 m   : {int(np.sum(finite > 50)):,}")
print(f"  cells > 100 m  : {int(np.sum(finite > 100)):,}")
# ============================================================
# 2) AVERAGE RECHARGE
# ============================================================
if isinstance(rch_spd, dict):
    rch_arrays = []
    for per in sorted(rch_spd.keys()):
        arr = np.array(rch_spd[per], dtype=float)

        # if recharge is given as 3D, use top layer / first slice
        if arr.ndim == 3:
            arr = arr[0]

        rch_arrays.append(arr)

    rch_all = np.array(rch_arrays, dtype=float)

else:
    arr = np.array(rch_spd, dtype=float)
    if arr.ndim == 3:
        rch_all = arr
    elif arr.ndim == 2:
        rch_all = arr[np.newaxis, :, :]
    else:
        raise ValueError("rch_spd has an unsupported shape")

avg_rch = np.nanmean(rch_all, axis=0)
avg_rch = np.where(idomain[0] > 0, avg_rch, np.nan)

# ============================================================
# 3) COLOR LIMITS
# ============================================================
dtw_vmin, dtw_vmax = robust_limits(avg_dtw, qlow=2, qhigh=98, symmetric=True)
dtw_vmin = -50.0    # artesian / head above land
dtw_vmax =  50.0    # max realistic DTW for Great Lakes Basin
dtw_norm = TwoSlopeNorm(vmin=dtw_vmin, vcenter=0.0, vmax=dtw_vmax)

rch_vmin, rch_vmax = robust_limits(avg_rch, qlow=2, qhigh=98, symmetric=False)

# ============================================================
# 4) BOUNDARY
# ============================================================
gdf_bdry = gpd.read_file(boundary_shp)
try:
    if getattr(gwf.modelgrid, "crs", None) is not None:
        gdf_bdry = gdf_bdry.to_crs(gwf.modelgrid.crs)
except Exception:
    pass

extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)

# ============================================================
# 5) PLOT
# ============================================================
# ============================================================
# 5) PLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

# ---- Average DTW
ax = axes[0]
im1 = ax.imshow(
    np.ma.masked_invalid(avg_dtw),
    origin="upper",
    extent=extent,
    cmap="RdBu",
    norm=dtw_norm,
)
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.text(
    0.02, 0.98, "a)",
    transform=ax.transAxes,
    ha="left", va="top",
    fontsize=14, fontweight="bold"
)
cbar1 = fig.colorbar(im1, ax=ax, shrink=0.82, extend="both")
cbar1.set_label("Average depth to water (m)")

# ---- Average Recharge
ax = axes[1]
im2 = ax.imshow(
    np.ma.masked_invalid(avg_rch),
    origin="upper",
    extent=extent,
    cmap="Blues",
    vmin=rch_vmin,
    vmax=rch_vmax,
)
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.text(
    0.02, 0.98, "b)",
    transform=ax.transAxes,
    ha="left", va="top",
    fontsize=14, fontweight="bold"
)
cbar2 = fig.colorbar(im2, ax=ax, shrink=0.82, extend="max")
cbar2.set_label("Average recharge (m/day)")

plt.show()

if save_fig:
    fig.savefig(out_fig, dpi=300, bbox_inches="tight")
    print("Saved:", out_fig)

# ============================================================
# 6) SUMMARY
# ============================================================
print("Average DTW min/max:", np.nanmin(avg_dtw), np.nanmax(avg_dtw))
print("Average recharge min/max:", np.nanmin(avg_rch), np.nanmax(avg_rch))

In [ ]:



# =========================================================
# 1) Paths to model outputs
# =========================================================
hds_path = Path(sim_ws) / f"{nameModel}.hds"
cbb_path = Path(sim_ws) / f"{nameModel}.cbb"
lst_path = Path(sim_ws) / f"{nameModel}.lst"

print("HDS exists:", hds_path.exists(), hds_path)
print("CBB exists:", cbb_path.exists(), cbb_path)
print("LST exists:", lst_path.exists(), lst_path)

assert hds_path.exists(), f"Head file not found: {hds_path}"


# =========================================================
# 2) Helper functions
# =========================================================












# =========================================================
# 3) Open head file and inspect saved times
# =========================================================
hdobj = flopy.utils.HeadFile(hds_path)
kstpkper_list = hdobj.get_kstpkper()
times = hdobj.get_times()

print("Number of saved times:", len(times))
print("kstpkper list:", kstpkper_list)

extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)
date_labels = get_date_labels(len(kstpkper_list), months=months if "months" in globals() else None)


# =========================================================
# 4) Quick diagnostics: raw vs masked ranges
# =========================================================
print("\n--- HEAD DIAGNOSTICS ---")
for i, kp in enumerate(kstpkper_list):
    h = hdobj.get_data(kstpkper=kp)
    h0_raw = np.array(h[0], dtype=float)
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)

    print(f"{date_labels[i]}:")
    print("   raw min/max   =", np.nanmin(h0_raw), np.nanmax(h0_raw))
    if np.isfinite(h0).any():
        print("   masked min/max=", np.nanmin(h0), np.nanmax(h0))
        print("   finite count  =", np.isfinite(h0).sum())
    else:
        print("   masked array has no finite values")


# =========================================================
# 5) Summary statistics table for layer 1 heads
# =========================================================
stats = []
for i, kp in enumerate(kstpkper_list):
    h = hdobj.get_data(kstpkper=kp)
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)

    stats.append({
        "stress_period": i + 1,
        "label": date_labels[i],
        "min_head_L1": np.nanmin(h0),
        "max_head_L1": np.nanmax(h0),
        "mean_head_L1": np.nanmean(h0),
        "std_head_L1": np.nanstd(h0),
    })

df_stats = pd.DataFrame(stats)
print("\n--- LAYER 1 HEAD STATS ---")
print(df_stats)


# =========================================================
# 6) Plot layer 1 heads for selected stress periods
# =========================================================
# choose which stress periods to show
plot_ids = [0, 5, 11]   # first, middle, last for a 12-month run
plot_ids = [i for i in plot_ids if i < len(kstpkper_list)]

head_arrays = []
for idx in plot_ids:
    h = hdobj.get_data(kstpkper=kstpkper_list[idx])
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)
    head_arrays.append(h0)

vmin, vmax = robust_limits(head_arrays, qlow=2, qhigh=98)
print("\nHead plot vmin/vmax:", vmin, vmax)

fig, axes = plt.subplots(1, len(plot_ids), figsize=(6 * len(plot_ids), 5), squeeze=False)

for ax, idx, arr in zip(axes.flat, plot_ids, head_arrays):
    im = ax.imshow(
        np.ma.masked_invalid(arr),
        origin="upper",
        extent=extent,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"Layer 1 head\n{date_labels[idx]}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Head (m)")

plt.tight_layout()
plt.show()


# =========================================================
# 7) Plot water table for selected stress periods
# =========================================================
wt_arrays = []
for idx in plot_ids:
    h = hdobj.get_data(kstpkper=kstpkper_list[idx])
    wt = get_water_table(h, idomain, huge=1e20)
    wt_arrays.append(wt)

vmin, vmax = robust_limits(wt_arrays, qlow=2, qhigh=98)
print("Water table plot vmin/vmax:", vmin, vmax)

fig, axes = plt.subplots(1, len(plot_ids), figsize=(6 * len(plot_ids), 5), squeeze=False)

for ax, idx, arr in zip(axes.flat, plot_ids, wt_arrays):
    im = ax.imshow(
        np.ma.masked_invalid(arr),
        origin="upper",
        extent=extent,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"Water table\n{date_labels[idx]}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Water table (m)")

plt.tight_layout()
plt.show()


# =========================================================
# 8) Plot head change from first to last stress period
# =========================================================
h_first = hdobj.get_data(kstpkper=kstpkper_list[0])
h_last  = hdobj.get_data(kstpkper=kstpkper_list[-1])

h_first_L1 = mask_model_array(h_first[0], idomain_layer=idomain[0], huge=1e20)
h_last_L1  = mask_model_array(h_last[0],  idomain_layer=idomain[0], huge=1e20)

dh = h_last_L1 - h_first_L1

finite_dh = dh[np.isfinite(dh)]
if finite_dh.size > 0:
    lim = np.nanpercentile(np.abs(finite_dh), 98)
else:
    lim = 1.0

plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(dh),
    origin="upper",
    extent=extent,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
)
plt.colorbar(shrink=0.8, label="Head change (m)")
plt.title(f"Layer 1 head change\n{date_labels[0]} to {date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.show()

print("\nHead change min/max (m):", np.nanmin(dh), np.nanmax(dh))



# =========================================================
# 10) Save final figures
# =========================================================
fig_dir = Path(r"D:\Users\abolmaal\modelling\Figs\Testing_6")
fig_dir.mkdir(exist_ok=True)

# final water table
wt_last = get_water_table(h_last, idomain, huge=1e20)

plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(wt_last),
    origin="upper",
    extent=extent,
    cmap="viridis",
    vmin=np.nanpercentile(wt_last[np.isfinite(wt_last)], 2),
    vmax=np.nanpercentile(wt_last[np.isfinite(wt_last)], 98),
)
plt.colorbar(shrink=0.8, label="Water table (m)")
plt.title(f"Final water table\n{date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.savefig(fig_dir / "final_water_table.png", dpi=300, bbox_inches="tight")
plt.show()

# head change
plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(dh),
    origin="upper",
    extent=extent,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
)
plt.colorbar(shrink=0.8, label="Head change (m)")
plt.title(f"Layer 1 head change\n{date_labels[0]} to {date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.savefig(fig_dir / "head_change_layer1.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved figures to:", fig_dir)

In [ ]:

# ------------------------------------------------------------
# READ HEAD OUTPUT AFTER A SUCCESSFUL RUN
# ------------------------------------------------------------
headfile = os.path.join(sim_ws, f"{nameModel}.hds")

if not os.path.exists(headfile):
    raise FileNotFoundError(
        f"Head file not found: {headfile}\n"
        "Make sure Cell 12 finished with Run success: True before plotting."
    )

hds = flopy.utils.HeadFile(headfile)

# all output times
times = hds.get_times()
print("Number of output times:", len(times))
print("First/last output time:", times[0], times[-1])

# read all heads into one array: shape = (ntime, nlay, nrow, ncol)
head = np.array([hds.get_data(totim=t) for t in times], dtype=float)

# dates matching TDIS monthly periods
dates = pd.date_range(start=START_DATE, periods=head.shape[0], freq="MS")

print("head shape:", head.shape)
print("dates:", dates[0], "to", dates[-1])

# optional plot index for selected maps
max_maps = min(6, head.shape[0])
plot_idx = np.linspace(0, head.shape[0] - 1, max_maps, dtype=int)

# quick sanity check
print("Head min/max:", np.nanmin(np.where(np.abs(head) < 1e20, head, np.nan)),
      np.nanmax(np.where(np.abs(head) < 1e20, head, np.nan)))

In [ ]:

# ============================================================
# SETTINGS
# ============================================================


out_fig_ts    = r"D:\Users\abolmaal\modelling\Figs\Testing_6\depthtowatertable.png"
out_fig_maps  = r"D:\Users\abolmaal\modelling\Figs\Testing_6\depthtowater_maps_blue_classes.png"
out_fig_final = r"D:\Users\abolmaal\modelling\Figs\Testing_6\depthtowater_final_blue_classes.png"

# ============================================================
# DEPTH-TO-GROUNDWATER COLOR CLASSES
# ============================================================
# Negative values will be removed/masked.
bounds = [0, 7, 25, 50, 100, 250, 500]

colors = [
    "#d9f2ff",  # Very shallow 0–7
    "#8be4ff",  # Shallow 7–25
    "#2fc7f0",  # Shallow medium 25–50
    "#10a8d2",  # Medium 50–100
    "#087f9f",  # Deep 100–250
    "#045a75",  # Very deep >250
]

dtw_cmap = ListedColormap(colors)
dtw_norm = BoundaryNorm(bounds, dtw_cmap.N)

# ============================================================
# FORCE RELOAD
# ============================================================
for var in ["head", "dates", "dtw_all", "dtw_final", "dtw_sel"]:
    if var in globals():
        del globals()[var]

head_path = os.path.join(sim_ws, f"{nameModel}.hds")
hds = bf.HeadFile(head_path)
times = hds.get_times()
head = np.array([hds.get_data(totim=t) for t in times], dtype=float)

print("Loaded head array:", head.shape)

dates = pd.date_range(start=START_DATE, periods=head.shape[0], freq="MS")
print("Dates:", dates[0].strftime("%Y-%m"), "to", dates[-1].strftime("%Y-%m"))
print("head periods:", head.shape[0], "| dates:", len(dates))

N_SHOW = min(N_SHOW_MAX, head.shape[0])

# ============================================================
# HELPERS
# ============================================================
top = np.array(gwf.dis.top.array, dtype=float)





# ============================================================
# MODEL BOUNDARY
# ============================================================
gdf_bdry = gpd.read_file(boundary_shp)

try:
    if getattr(gwf.modelgrid, "crs", None) is not None:
        gdf_bdry = gdf_bdry.to_crs(gwf.modelgrid.crs)
except Exception:
    pass

plot_idx = np.linspace(0, head.shape[0] - 1, N_SHOW, dtype=int)

# ============================================================
# 1) FINAL DEPTH-TO-WATER MAP
# ============================================================
dtw_final = get_depth_to_water(head[-1], idomain, top, clip_negative=True)
dtw_final_plot = np.clip(dtw_final, bounds[0], bounds[-1])

fig, ax = plt.subplots(figsize=(9, 7))

pmv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
pcm = pmv.plot_array(
    dtw_final_plot,
    cmap=dtw_cmap,
    norm=dtw_norm,
)

gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)

ax.grid(False)
ax.set_title(f"Final depth to groundwater\n{dates[-1].strftime('%Y-%m')}")
ax.set_xlabel("X")
ax.set_ylabel("Y")

add_north_arrow(ax)
add_scale_bar(ax, delr, delc, length_km=SCALEBAR_KM, loc="lower left")
add_dtw_colorbar(fig, pcm, ax, bounds)

plt.tight_layout()
plt.savefig(out_fig_final, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 2) DEPTH-TO-WATER MAPS AT DIFFERENT TIMES
# ============================================================
dtw_sel = [
    get_depth_to_water(head[i], idomain, top, clip_negative=True)
    for i in plot_idx
]

nplots = len(plot_idx)
ncols = 3
nrows = int(np.ceil(nplots / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(5 * ncols, 4.8 * nrows),
    constrained_layout=True,
)

axes = np.atleast_1d(axes).ravel()
panel_labels = ["a", "b", "c", "d", "e", "f"]

for k, (ax, i, arr) in enumerate(zip(axes, plot_idx, dtw_sel)):
    arr_plot = np.clip(arr, bounds[0], bounds[-1])

    pmv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    pcm = pmv.plot_array(
        arr_plot,
        cmap=dtw_cmap,
        norm=dtw_norm,
    )

    gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.7)

    ax.grid(False)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_title(
        f"({panel_labels[k]}) {dates[i].strftime('%Y-%m')}",
        loc="left",
        fontweight="bold",
    )

    add_north_arrow(ax, x=0.90, y=0.88, size=0.09)
    add_scale_bar(ax, delr, delc, length_km=SCALEBAR_KM, loc="lower left")

for j in range(len(plot_idx), len(axes)):
    axes[j].axis("off")

add_dtw_colorbar(fig, pcm, axes.tolist(), bounds)

plt.savefig(out_fig_maps, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 3) BASIN-AVERAGE DEPTH TO WATER THROUGH TIME
# ============================================================
dtw_all = np.array([
    get_depth_to_water(head[i], idomain, top, clip_negative=True)
    for i in range(head.shape[0])
])

dtw_clipped = np.clip(dtw_all, bounds[0], bounds[-1])

mean_dtw = np.nanmean(dtw_clipped, axis=(1, 2))
p05_dtw  = np.nanpercentile(dtw_clipped, 5, axis=(1, 2))
p25_dtw  = np.nanpercentile(dtw_clipped, 25, axis=(1, 2))
p75_dtw  = np.nanpercentile(dtw_clipped, 75, axis=(1, 2))
p95_dtw  = np.nanpercentile(dtw_clipped, 95, axis=(1, 2))

fig, ax = plt.subplots(figsize=(11, 4.8))

ax.fill_between(dates, p25_dtw, p75_dtw, alpha=0.25, color="steelblue", label="25–75 percentile")
ax.fill_between(dates, p05_dtw, p95_dtw, alpha=0.15, color="steelblue", label="5–95 percentile")
ax.plot(dates, mean_dtw, linewidth=2.0, color="steelblue", label="Mean depth to groundwater")

ax.set_xlabel("Date")
ax.set_ylabel("Depth to groundwater (m below ground level)")
ax.set_title("Basin depth to groundwater through time")
ax.legend()

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2 if len(dates) <= 24 else 12))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(out_fig_ts, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# SUMMARY
# ============================================================
print("Saved maps:")
print(out_fig_final)
print(out_fig_maps)
print(out_fig_ts)

active = idomain[0] > 0
valid_final = dtw_final[active]

print("\nFinal DTW stats, active cells, negative values removed:")
for pct in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  p{pct:2d}: {np.nanpercentile(valid_final, pct):8.1f} m")

print(f"Negative DTW cells removed from final map: {np.sum(get_depth_to_water(head[-1], idomain, top, clip_negative=False)[active] < 0):,}")
print(f"Color classes: {bounds}")

# Compare the Simulation and observartions 

In [ ]:
# ============================================================
# Compare MODFLOW 6 Simulated Heads with Well Observations
#
# SWL = static water level / depth to water below land surface
# Observed head:   obs_head_m  = land_elev_m - SWL_m
# Simulated head:  temporal mean across all stress periods
# ============================================================


# ============================================================
# 1. SIMULATED HEADS: temporal mean over all stress periods
# ============================================================
# head shape: (nper, nlay, nrow, ncol)
# Mean over all periods is more robust than using a single timestep
# for calibration when observation dates are unknown.
h = np.mean(head, axis=0)   # shape: (nlay, nrow, ncol)

print("Mean head shape:", h.shape)
print("Simulation period:", times[0], "to", times[-1])


# ============================================================
# 2. READ WELL DATA
# ============================================================
target_crs = "EPSG:3174"

print("Available layers:")
print(pyogrio.list_layers(wells_gdb_path))

wells = gpd.read_file(
    wells_gdb_path,
    layer=WELL_LAYER,
    engine="pyogrio"
)

boundary = gpd.read_file(boundary_shp)

wells    = wells.to_crs(target_crs)
boundary = boundary.to_crs(target_crs)

needed_cols = [
    "WELL_ID", "LAT", "LON",
    "SWL", "WELL_DEPTH",
    "SCREEN_FRM", "SCREEN_TO",
    "AQ_TYPE", "WELL_TYPE",
    "geometry",
]
existing_cols = [c for c in needed_cols if c in wells.columns]
wells = wells[existing_cols].copy()

for col in ["SWL", "WELL_DEPTH", "SCREEN_FRM", "SCREEN_TO"]:
    if col in wells.columns:
        wells[col] = pd.to_numeric(wells[col], errors="coerce")

wells = wells.dropna(subset=["SWL", "geometry"]).copy()
print("Original wells after removing missing SWL:", len(wells))


# ============================================================
# 3. CLIP WELLS TO MODEL BOUNDARY
# ============================================================
boundary_union = boundary.geometry.union_all()
wells = wells[wells.geometry.within(boundary_union)].copy()
print("Wells inside model boundary:", len(wells))


# ============================================================
# 4. CLEAN AND CONVERT SWL / WELL DEPTH (feet -> metres)
# ============================================================
wells["SWL_m"] = wells["SWL"] * FT_TO_M

if "WELL_DEPTH" in wells.columns:
    wells["WELL_DEPTH_m"] = wells["WELL_DEPTH"] * FT_TO_M
if "SCREEN_FRM" in wells.columns:
    wells["SCREEN_FRM_m"] = wells["SCREEN_FRM"] * FT_TO_M
if "SCREEN_TO" in wells.columns:
    wells["SCREEN_TO_m"] = wells["SCREEN_TO"] * FT_TO_M

wells = wells[
    (wells["SWL_m"] >= 0) &
    (wells["SWL_m"] < 150)
].copy()
print("Wells after SWL cleaning:", len(wells))


# ============================================================
# 5. GET MODEL ROW/COL -- with out-of-bounds guard
# ============================================================
wells["x_3174"] = wells.geometry.x
wells["y_3174"] = wells.geometry.y

rows, cols, valid = [], [], []
for x, y in zip(wells["x_3174"], wells["y_3174"]):
    try:
        r, c = gwf.modelgrid.intersect(x, y)
        if 0 <= int(r) < nrow and 0 <= int(c) < ncol:
            rows.append(int(r)); cols.append(int(c)); valid.append(True)
        else:
            rows.append(-1); cols.append(-1); valid.append(False)
    except Exception:
        rows.append(-1); cols.append(-1); valid.append(False)

wells["row"]      = rows
wells["col"]      = cols
wells["_in_grid"] = valid
wells = wells[wells["_in_grid"]].drop(columns="_in_grid").copy()
print("Wells inside model grid:", len(wells))

# Land surface elevation (= model top at that cell)
wells["land_elev_m"] = top[
    wells["row"].astype(int),
    wells["col"].astype(int)
]

# Observed groundwater head = land surface minus static water level
wells["obs_head_m"] = wells["land_elev_m"] - wells["SWL_m"]

wells = wells[
    (wells["obs_head_m"] > 0) &
    (wells["obs_head_m"] < 700)
].copy()
print("Wells after observed-head cleaning:", len(wells))


# ============================================================
# 6. ASSIGN MODEL LAYER USING SCREEN OR WELL DEPTH
# ============================================================
if "SCREEN_FRM_m" in wells.columns and "SCREEN_TO_m" in wells.columns:
    wells["screen_mid_m"] = (wells["SCREEN_FRM_m"] + wells["SCREEN_TO_m"]) / 2
else:
    wells["screen_mid_m"] = np.nan

if "WELL_DEPTH_m" in wells.columns:
    wells["screen_mid_m"] = wells["screen_mid_m"].fillna(wells["WELL_DEPTH_m"] / 2)

wells["screen_mid_m"] = wells["screen_mid_m"].fillna(5.0)

layers = []
for _, r in wells.iterrows():
    i_w = int(r["row"])
    j_w = int(r["col"])
    screen_mid_elev = r["land_elev_m"] - r["screen_mid_m"]
    assigned_layer  = 0
    for k in range(botm.shape[0]):
        layer_top = top[i_w, j_w] if k == 0 else botm[k - 1, i_w, j_w]
        layer_bot = botm[k, i_w, j_w]
        if layer_bot <= screen_mid_elev <= layer_top:
            assigned_layer = k
            break
    layers.append(assigned_layer)
wells["layer"] = layers


# ============================================================
# 7. DROP WELLS IN INACTIVE CELLS (idomain <= 0)
# ============================================================
# MF6 inactive cells have no valid simulated head
wells["_idomain"] = [
    int(idomain[int(r["layer"]), int(r["row"]), int(r["col"])])
    for _, r in wells.iterrows()
]
wells = wells[wells["_idomain"] > 0].drop(columns="_idomain").copy()
print("Wells in active cells:", len(wells))


# ============================================================
# 8. BUILD OBSERVATION TABLE AND SAVE
# ============================================================
wells = wells.reset_index(drop=True)

if "WELL_ID" in wells.columns:
    wells["well_id"] = wells["WELL_ID"].astype(str)
else:
    wells["well_id"] = [f"WELL_{i:06d}" for i in range(1, len(wells) + 1)]

keep_cols = [
    "well_id", "LAT", "LON", "x_3174", "y_3174",
    "land_elev_m", "SWL", "SWL_m", "obs_head_m",
    "WELL_DEPTH", "WELL_DEPTH_m",
    "SCREEN_FRM", "SCREEN_TO", "SCREEN_FRM_m", "SCREEN_TO_m",
    "screen_mid_m", "AQ_TYPE", "WELL_TYPE",
    "layer", "row", "col", "geometry",
]
keep_cols = [c for c in keep_cols if c in wells.columns]
obs_table = wells[keep_cols].copy()

os.makedirs(obs_out_dir, exist_ok=True)
obs_table.drop(columns="geometry", errors="ignore").to_csv(out_obs_csv, index=False)
print("Saved observation table:", out_obs_csv)


# ============================================================
# 9. EXTRACT SIMULATED HEAD (temporal mean)
# ============================================================
comparison = obs_table.copy()

k_arr = comparison["layer"].astype(int).values
i_arr = comparison["row"].astype(int).values
j_arr = comparison["col"].astype(int).values

comparison["sim_head_m"] = h[k_arr, i_arr, j_arr]

# Mask MF6 dry-cell sentinel (1e30) -- use abs threshold, not exact equality
comparison.loc[np.abs(comparison["sim_head_m"]) >= 1e20, "sim_head_m"] = np.nan

comparison = comparison.dropna(subset=["obs_head_m", "sim_head_m"]).copy()

comparison = comparison[
    (comparison["obs_head_m"] > 0) &
    (comparison["obs_head_m"] < 700) &
    (comparison["sim_head_m"] > 0) &
    (comparison["sim_head_m"] < 700)
].copy()


# ============================================================
# 10. HEAD COMPARISON STATISTICS
# ============================================================
comparison["residual_m"]  = comparison["sim_head_m"] - comparison["obs_head_m"]
comparison["abs_error_m"] = comparison["residual_m"].abs()

rmse = np.sqrt(np.mean(comparison["residual_m"] ** 2))
mae  = np.mean(comparison["abs_error_m"])
bias = np.mean(comparison["residual_m"])

print("Number of comparison wells:", len(comparison))
print(f"Head RMSE: {rmse:.2f} m")
print(f"Head MAE : {mae:.2f} m")
print(f"Head Bias: {bias:.2f} m")


# ============================================================
# 11. DEPTH-TO-WATER COMPARISON
# ============================================================
comparison["obs_dtw_m"] = comparison["SWL_m"]
comparison["sim_dtw_m"] = comparison["land_elev_m"] - comparison["sim_head_m"]

comparison = comparison[
    (comparison["obs_dtw_m"] >= 0) & (comparison["obs_dtw_m"] < 150) &
    (comparison["sim_dtw_m"] >= 0) & (comparison["sim_dtw_m"] < 150)
].copy()

comparison["dtw_residual_m"] = comparison["sim_dtw_m"] - comparison["obs_dtw_m"]

dtw_rmse = np.sqrt(np.mean(comparison["dtw_residual_m"] ** 2))
dtw_mae  = np.mean(np.abs(comparison["dtw_residual_m"]))
dtw_bias = np.mean(comparison["dtw_residual_m"])

print(f"DTW RMSE: {dtw_rmse:.2f} m")
print(f"DTW MAE : {dtw_mae:.2f} m")
print(f"DTW Bias: {dtw_bias:.2f} m")


# ============================================================
# 12. SAVE COMPARISON TABLE
# ============================================================
os.makedirs(compare_out_dir, exist_ok=True)
comparison.drop(columns="geometry", errors="ignore").to_csv(out_compare_csv, index=False)
print("Saved comparison table:", out_compare_csv)


# ============================================================
# 13. PLOT OBSERVED VS SIMULATED HEAD
# ============================================================
plot_df = comparison.copy()
if len(plot_df) > 100000:
    plot_df = plot_df.sample(100000, random_state=42)

plt.figure(figsize=(6, 6))
plt.scatter(plot_df["obs_head_m"], plot_df["sim_head_m"], s=5, alpha=0.25)

min_val = min(comparison["obs_head_m"].min(), comparison["sim_head_m"].min())
max_val = max(comparison["obs_head_m"].max(), comparison["sim_head_m"].max())
plt.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1)

plt.xlabel("Observed groundwater head (m)")
plt.ylabel("Simulated groundwater head (m)")
plt.title("Observed vs Simulated Groundwater Head")
plt.grid(True, alpha=0.3)
plt.text(
    0.05, 0.95,
    f"RMSE = {rmse:.2f} m\nMAE = {mae:.2f} m\nBias = {bias:.2f} m\nn = {len(comparison):,}",
    transform=plt.gca().transAxes, verticalalignment="top",
    bbox=dict(facecolor="white", edgecolor="black", alpha=0.8)
)
plt.tight_layout()
plt.savefig(out_compare_fig, dpi=300)
plt.show()


# ============================================================
# 14. PLOT OBSERVED VS SIMULATED DEPTH TO WATER
# ============================================================
plt.figure(figsize=(6, 6))
plt.scatter(plot_df["obs_dtw_m"], plot_df["sim_dtw_m"], s=5, alpha=0.25)

min_val = min(comparison["obs_dtw_m"].min(), comparison["sim_dtw_m"].min())
max_val = max(comparison["obs_dtw_m"].max(), comparison["sim_dtw_m"].max())
plt.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1)

plt.xlabel("Observed depth to water, SWL (m)")
plt.ylabel("Simulated depth to water (m)")
plt.title("Observed vs Simulated Depth to Water")
plt.grid(True, alpha=0.3)
plt.text(
    0.05, 0.95,
    f"RMSE = {dtw_rmse:.2f} m\nMAE = {dtw_mae:.2f} m\nBias = {dtw_bias:.2f} m\nn = {len(comparison):,}",
    transform=plt.gca().transAxes, verticalalignment="top",
    bbox=dict(facecolor="white", edgecolor="black", alpha=0.8)
)
plt.tight_layout()
plt.savefig(out_dtw_fig, dpi=300)
plt.show()

print("Saved figures:")
print(out_compare_fig)
print(out_dtw_fig)


In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, aspect="equal")

# Next we create an instance of the ModelMap class
modelmap = flopy.plot.PlotMapView(model=gwf, ax=ax)

ghb_quadmesh = modelmap.plot_bc(name="ghb", plotAll=True)
riv_quadmesh = modelmap.plot_bc(name="riv", plotAll=True)
linecollection = modelmap.plot_grid()
contours = modelmap.contour_array(h[0])